<a href="https://colab.research.google.com/github/shreyanalam/weather-trip-planner-agent/blob/main/weather-trip-planner-agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# AI code review
import google.generativeai as genai
import gradio as gr

genai.configure(api_key=" ")
def review_code(code,input_programming_language,output_programming_languages):
  prompt=f"""
  Review this code from input_programming_language to output_programming_languages.
  provide:
  1.Summary
  2.Bugs
  3.Performance Improvements
  4.Code style
  5.Security issues
  6.Improved code
  7.Rating out of 10
  Code:
  {code}
  Input Programming Language:
  {input_programming_language}
  Output Programming Language:
  {output_programming_languages}
  """
  model = genai.GenerativeModel("gemini-2.5-flash")
  response = model.generate_content(prompt)
  return response.text
demo=gr.Interface(
    fn=review_code,
    inputs=[
    gr.Textbox(label="Code"),
    gr.Dropdown(
        ["python","java","html","css"],
        label="input_programming_language"
    ),
    gr.Dropdown(
        ["python","java","html","css"],
        label="output_programming_language"
    ),
    ],
    outputs="markdown",
    title=" AI Code Review (Gemini)"
)
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://361eb86f8a2f7a6c28.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



KeyboardInterrupt



In [ ]:
# gen ai tutor
from google import genai
import gradio as gr
client=genai.Client(api_key=" ")
SYSTEM_PROMPT="""
  You are a studymate AI and very intelligent tutor and educational assistant
  Instructions:
- Answer study-related questions clearly and accurately.
- Explain concepts in simple and easy-to-understand language.
- Provide step-by-step explanations whenever appropriate.
- Use examples to improve understanding.
- If programming code is requested, provide complete and well-commented code.
- Explain code line by line if requested.
- Present information using headings, bullet points, or tables whenever helpful.
- If solving numerical problems, show all calculation steps.
- If multiple solutions exist, explain the best one first.
- If you are unsure of an answer, clearly state that instead of guessing.
- Maintain a polite, friendly, and professional tone.
- Format every response neatly for easy reading.
"""
chat=client.chats.create(
    model="gemini-2.5-flash",
    config={
        "system_instruction":SYSTEM_PROMPT
    }
)

def respond(message,history):
  global chat
  response=chat.send_message(message)
  return response.text
def clear_chat():
  global chat

  chat=client.chats.create(
      model="gemini-2.5-flash",
      config={
          "system_instruction":SYSTEM_PROMPT
      }
  )

  return [],""
with gr.Blocks(title="AI STUDY TUTORR") as demo:
  gr.Markdown("# study Mate AI")
  gr.Markdown("# Ask any study related question")
  chatbot=gr.ChatInterface(
      fn=respond,
  )
  clear_btn=gr.Button("Clear Chat")
  clear_btn.click(
      fn=clear_chat,
      outputs=[chatbot.chatbot,chatbot.textbox]
  )
  demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://48e3c51e3936a32bbc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
#weather agent
!pip install -q gradio google-genai requests
import gradio as gr
import requests
from google import genai
client=genai.Client(api_key=" ")
def get_coordinates(city):
  url=f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1"
  response=requests.get(url,timeout=10)
  data=response.json()
  if "results" not in data:
    return None
  lat=data["results"][0]["latitude"]
  lon=data["results"][0]["longitude"]
  return lat,lon
def get_weather(lat,lon):
  url=(
      f"https://api.open-meteo.com/v1/forecast?"
      f"latitude={lat}&longitude={lon}"
      f"&current=temperature_2m,relative_humidity_2m,"
      f"wind_speed_10m,weather_code"
  )
  response=requests.get(url,timeout=10)
  data=response.json()
  return data["current"]
def weather_agent(query):
  prompt=f"""
extract only the city name from sentence below.
Sentence:
{query}
Return only the city name.
"""
  city=client.models.generate_content(
      model="gemini-2.5-pro",
      contents=prompt
  ).text.strip()
  location=get_coordinates(city)
  if location is None:
    return "City not found."
  lat,lon=location
  weather=get_weather(lat,lon)
  final_prompt=f"""
The user asked:
{query}
Weather Data
temperature:{weather['temperature_2m']}c
humidity:{weather['relative_humidity_2m']}%
wind_speed:{weather['wind_speed_10m']}km/h
weather_code:{weather['weather_code']}
Explain the weather in a friendly way
"""
  answer=client.models.generate_content(
      model="gemini-2.5-pro",
      contents=final_prompt
  )
  return answer.text

demo = gr.Interface(
    fn=weather_agent,
    inputs=gr.Textbox(
        label="Ask About Weather",
        placeholder="Example: What's the weather in Hyderabad?"
    ),
    outputs=gr.Textbox(label="Weather Report"),
    title="🌦️ AI Weather Agent",
    description="Ask about the weather in any city."
)

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://baf07d019288e2972d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
#rag
!pip install -q gradio chromadb pypdf sentence-transformers google-generativeai
import gradio as gr
import chromadb
import google.generativeai as genai
import uuid
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
genai.configure(api_key=" ")
llm=genai.GenerativeModel("gemini-2.5-flash")
print("Loading embedding model...")
embedder=SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded!")
client=chromadb.Client()
collection=client.get_or_create_collection("rag_docs")
def extract_text(pdf):
  reader=PdfReader(pdf.name)
  text = ""
  for page in reader.pages:
    page_text=page.extract_text()
    if page_text:
      text+=page_text+"\n"
  return text
def chunk_text(text,size=300):
  return [text[i:i+size] for i in range(0,len(text),size)]
def upload_pdf(pdf):
  global collection
  text=extract_text(pdf)
  chunks=chunk_text(text)
  print(f"Chunks Created: {len(chunks)}")
  embeddings=embedder.encode(chunks).tolist()
  #reset collection for new PDF
  try:
    client.delete_collection("rag_docs")
  except:
    pass
  collection=client.get_or_create_collection("rag_docs")
  collection.add(
      documents=chunks,
      embeddings=embeddings,
      ids=[str(uuid.uuid4())for _ in chunks]
      )
  return f" PDF Indexed Successfully ({len(chunks)} chunks)"
def ask(question):
  query_embedding=embedder.encode(question).tolist()
  result=collection.query(
      query_embeddings=[query_embedding],
      n_results=3
  )
  context="\n".join(result["documents"][0])
  context=context[:4000]
  prompt=f"""
Answer ONLY using the context below
If the answer is not found, say:
'I couldn't find the answer in the document.'
Context:
{context}
Question:
{question}
"""
  response=llm.generate_content(prompt)
  return response.text
with gr.Blocks() as demo:
    gr.Markdown("# 📄 RAG PDF Chatbot")

    pdf = gr.File(label="Upload PDF")
    upload_btn = gr.Button("Index PDF")
    status = gr.Textbox(label="Status")

    upload_btn.click(upload_pdf, inputs=pdf, outputs=status)

    question = gr.Textbox(label="Ask a Question")
    ask_btn = gr.Button("Ask")
    answer = gr.Textbox(label="Answer", lines=10)

    ask_btn.click(ask, inputs=question, outputs=answer)

demo.launch(debug=True)

In [ ]:
#ai voice assisstant
import gradio as gr
from google import genai
import speech_recognition as sr
from gtts import gTTS
import tempfile

client=genai.Client(api_key=" ")

#Speech to Text
def speech_to_text(audio):
  recognizer=sr.Recognizer()

  with sr.AudioFile(audio) as source:
    audio_data=recognizer.record(source)
  try:
    return recognizer.recognize_google(audio_data)
  except:
    return "Sorry, I couldn't understand."

def ask_gemini(text):
  response=client.models.generate_content(
      model="gemini-2.5-flash",
      contents=text
  )
  return response.text

#text to speech
def text_to_speech(text):
  tts=gTTS(text)
  path=tempfile.NamedTemporaryFile(delete=False,suffix=".mp3").name
  tts.save(path)
  return path

#main Function
def voice_assisstant(audio):
  user_text=speech_to_text(audio)
  ai_reply=ask_gemini(user_text)
  voice=text_to_speech(ai_reply)

  return user_text,ai_reply,voice

demo=gr.Interface(
    fn=voice_assisstant,
    inputs=gr.Audio(type="filepath",label="Speak"),
    outputs=[
        gr.Textbox(label="You Said"),
        gr.Textbox(label="Gemini"),
        gr.Textbox(label="Voice Response")
    ],
    title="Simple AI voice Assisstance."
)

demo.launch(debug=True,share=True)

In [ ]:
#speech to text
# From Speech to text

import gradio as gr
from google import genai

#Gemini API Key
client=genai.Client(api_key=" ")

def transcribe(audio_path):
  if audio_path is None:
    return "Please record some audio."

    #upload recorded audio
    audio_file=client.files.upload(file=audio_path)

    #transcribe
    response=client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[
            "Transcribe this audio exactly as spoken. Do not summarize. Include punctuation.",
            audion_file,
        ],
    )
    return response.text

with gr.Blocks(title="Gemini Audio Transcription") as demo:
  gr.Markdown("# Audio Transcription with Gemini")

  audio=gr.Audio(
      sources=["microphone"],
      type="filepath",
      label="Record Your voice"
  )

  output=gr.Textbox(
     label="Transcript",
     lines=10
  )

  btn=gr.Button("Transcribe")

  btn.click(
      fn=transcribe,
      inputs=audio,
      outputs=output
  )
demo.launch()

In [ ]:
#text to speech
# Text to speech
from gtts import gTTS
import gradio as gr

def text_to_speech(text):
  tts=gTTS(text=text,lang="en")
  output_file="output.mp3"
  tts.save(output_file)
  return output_file

demo=gr.Interface(fn=text_to_speech,
                  inputs=gr.Textbox(lines=5,label="Enter Text"),
                  outputs=gr.Audio(label="Speech"),
                  title="Text-to-Speech"
                  )
demo.launch()

In [ ]:
#gemini document analyzer
import gradio as gr
import os
from google import genai
from google.genai import types
from gtts import gTTS
import tempfile

# ====================================
# Gemini API Key
# ====================================
client = genai.Client(
    api_key=os.getenv("Gemini_API")
)

# ====================================
# Core Task Functions
# ====================================
def extract_text(file):
    if file is None:
        return "Please upload a file first.", None

    uploaded_file_ref = client.files.upload(file=file.name)
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[uploaded_file_ref, "Extract all text exactly as written."]
    )
    return response.text, None


def summarize(file):
    if file is None:
        return "Please upload a file first.", None

    uploaded_file_ref = client.files.upload(file=file.name)
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[uploaded_file_ref, "Summarize this document in simple language."]
    )
    return response.text, None


def answer_question(file, question):
    if file is None:
        return "Please upload a file first.", None
    if not question.strip():
        return "Please enter a question.", None

    uploaded_file_ref = client.files.upload(file=file.name)
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[
            uploaded_file_ref,
            f"Answer the following question using ONLY this document.\n\nQuestion:\n{question}"
        ]
    )
    return response.text, None


def audio_summary(file):
    if file is None:
        return "Please upload a file first.", None

    uploaded_file_ref = client.files.upload(file=file.name)
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[uploaded_file_ref, "Summarize this document."]
    )
    summary = response.text

    # Generate Audio file
    audio_file = tempfile.NamedTemporaryFile(
        suffix=".mp3",
        delete=False
    ).name
    gTTS(summary).save(audio_file)

    return summary, audio_file


# ====================================
# UI Layout (No Dropdown Layout)
# ====================================
with gr.Blocks() as demo:

    gr.Markdown("# 📄 Gemini Document Analyzer")

    with gr.Row():
        # Left Side: File Upload Window
        with gr.Column(scale=1):
            file = gr.File(label="Upload PDF/Image")

        # Right Side: Structured Actions Panels
        with gr.Column(scale=2):
            gr.Markdown("### 🛠️ Choose an Action")

            with gr.Row():
                btn_extract = gr.Button("🔍 Extract Text", variant="secondary")
                btn_summarize = gr.Button("📝 Summarize Document", variant="secondary")
                btn_audio = gr.Button("🔊 Generate Audio Summary", variant="secondary")

            gr.Markdown("---")
            gr.Markdown("### ❓ Document Q&A")
            question = gr.Textbox(
                label="Ask a specific question about the document",
                placeholder="Type your question here..."
            )
            btn_qa = gr.Button("💡 Answer Question", variant="primary")

    gr.Markdown("---")
    gr.Markdown("### 📊 Outputs")

    with gr.Row():
        output = gr.Textbox(label="Text Output", lines=12, scale=2)
        audio = gr.Audio(label="Audio Playback", scale=1)

    # ====================================
    # Button Event Bindings
    # ====================================
    btn_extract.click(
        fn=extract_text,
        inputs=[file],
        outputs=[output, audio]
    )

    btn_summarize.click(
        fn=summarize,
        inputs=[file],
        outputs=[output, audio]
    )

    btn_audio.click(
        fn=audio_summary,
        inputs=[file],
        outputs=[output, audio]
    )

    btn_qa.click(
        fn=answer_question,
        inputs=[file, question],
        outputs=[output, audio]
    )

demo.launch(
    server_name="0.0.0.0",
    server_port=int(os.environ.get("PORT", 7860)),
    debug=True
)

In [ ]:
!pip -q install groq duckduckgo-search gradio requests
from groq import Groq
from duckduckgo_search import DDGS
import requests, gradio as gr

client = Groq(api_key="")

MODEL = "llama-3.3-70b-versatile"
def weather(city):

    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name":city,"count":1}
    ).json()

    if "results" not in geo:
        return "City not found"

    p = geo["results"][0]

    data = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude":p["latitude"],
            "longitude":p["longitude"],
            "current":"temperature_2m"
        }
    ).json()

    return f"{city} Temperature : {data['current']['temperature_2m']}°C"
def search(query):

    with DDGS() as d:

        r = list(d.text(query,max_results=3))

    return "\n".join(i["title"] for i in r)
def agent(question):

    decision = client.chat.completions.create(

        model=MODEL,

        messages=[

            {
                "role":"system",
                "content":
                """
                Reply with only one word:

                weather
                search
                both
                none
                """
            },

            {
                "role":"user",
                "content":question
            }

        ]

    ).choices[0].message.content.lower()

    info = ""

    if "weather" in decision or "both" in decision:

        city = client.chat.completions.create(

            model=MODEL,

            messages=[
                {
                    "role":"user",
                    "content":f"Extract only the city name from: {question}"
                }
            ]

        ).choices[0].message.content.strip()

        info += weather(city) + "\n\n"

    if "search" in decision or "both" in decision:

        info += search(question)

    answer = client.chat.completions.create(

        model=MODEL,

        messages=[

            {
                "role":"system",
                "content":"Answer using the information provided."
            },

            {
                "role":"user",
                "content":f"Question:\n{question}\n\nInformation:\n{info}"
            }

        ]

    )

    return answer.choices[0].message.content
print(agent("Should I carry an umbrella for Goa and suggest places to visit?"))
gr.ChatInterface(
    fn=lambda x, h: agent(x),
    title="Weather + Trip Planner"
).launch(
    share=True,
    debug=False,
)